# UdaPlay: Video Game Research & Analytics Assistant
## Part 01 - Local Vector Database & Offline RAG Foundation

### Project Overview & Objectives
This notebook establishes the local Retrieval-Augmented Generation (RAG) vector database for **UdaPlay**.

The underlying knowledge base is built from curated video game entries in the `games/` folder. Each document contains rich metadata describing the game's title, platform, genre, publisher, release year, and gameplay summary.

**Core Objectives:**
1. Load and parse the video game catalog from the `games/` directory.
2. Initialize a persistent **ChromaDB** client storing embeddings in `./chromadb`.
3. Configure OpenAI embeddings using the environment credentials (`OPENAI_API_KEY`, `OPENAI_BASE_URL`).
4. Ingest and embed the games into a Chroma collection named `udaplay`.
5. Define a structured Pydantic data model (`GameDocument`) for safe, typed record retrieval.
6. Validate document lookup and analyze catalog coverage across platforms and release years.

### 1. Environment Setup & Library Imports

In [1]:
import os
import json
from typing import Annotated, List, Dict, Optional
import pandas as pd
from pydantic import BaseModel, Field

import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv, find_dotenv
from rich.console import Console
from rich.markdown import Markdown

console = Console()

In [2]:
# Load environment variables from .env file
load_dotenv(find_dotenv(), override=True)

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set. Please configure .env file."
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set. Please configure .env file."
print("Environment configuration successfully verified.")

Environment configuration successfully verified.


### 2. Persistent ChromaDB Client Initialization

In [3]:
# Instantiate persistent ChromaDB client
chroma_client = chromadb.PersistentClient(path="chromadb")

### 3. Embedding Function & Collection Setup

In [4]:
# Configure OpenAI embedding function using configured API credentials
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=os.getenv("OPENAI_BASE_URL"),
)

In [5]:
# Initialize or retrieve the 'udaplay' collection
existing_collections = [c.name for c in chroma_client.list_collections()]

if "udaplay" not in existing_collections:
    collection = chroma_client.create_collection(
        name="udaplay",
        embedding_function=embedding_fn
    )
else:
    collection = chroma_client.get_collection(
        name="udaplay",
        embedding_function=embedding_fn
    )

### 4. Document Processing & Ingestion

In [6]:
# Load and index video game JSON records
data_dir = "games"
json_files = sorted([f for f in os.listdir(data_dir) if f.endswith(".json")])

for file_name in json_files:
    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game_info = json.load(f)

    # Format semantic search document content
    doc_content = f"[{game_info['Platform']}] {game_info['Name']} ({game_info['YearOfRelease']}) - {game_info['Description']}"
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[doc_content],
        metadatas=[game_info]
    )

print(f"Indexed {len(json_files)} game entries into ChromaDB collection 'udaplay'.")

Indexed 48 game entries into ChromaDB collection 'udaplay'.


### 5. Document Model Definition & Collection Inspection

In [7]:
# Verify stored records in collection
all_records = collection.get()
print(f"Collection fields: {list(all_records.keys())}")

Collection fields: ['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas']


In [8]:
class GameDocument(BaseModel):
    """Strongly-typed Pydantic model for game records retrieved from vector store."""
    Platform: Annotated[str, Field(description="Gaming platform (PC, PlayStation, Nintendo, Xbox)")]
    Name: Annotated[str, Field(description="Official game title")]
    YearOfRelease: Annotated[int, Field(description="Release calendar year")]
    Description: Annotated[str, Field(description="Overview of gameplay and storyline")]

In [9]:
# Convert retrieved metadata dictionaries into GameDocument models
metadatas = all_records.get('metadatas', [])
documents = [GameDocument(**meta) for meta in metadatas]

In [10]:
console.print(f"{len(documents)} retrieved from the database. Examples: ", style='bold yellow')
documents[:3]

48 retrieved from the database. Examples: 

[GameDocument(Platform='PlayStation 1', Name='Gran Turismo', YearOfRelease=1997, Description='A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.'),
 GameDocument(Platform='PlayStation 2', Name='Grand Theft Auto: San Andreas', YearOfRelease=2004, Description="An expansive open-world game set in the fictional state of San Andreas, following the story of Carl 'CJ' Johnson."),
 GameDocument(Platform='PlayStation 3', Name='Gran Turismo 5', YearOfRelease=2010, Description='A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.')]

In [11]:
def inspect_game_record(collection_name: str, record_id: int) -> Optional[Dict]:
    """
    Fetch and display a specific document record from ChromaDB by identifier.
    
    Args:
        collection_name (str): Target ChromaDB collection name.
        record_id (int): Record integer identifier (e.g., 1 for '001').
        
    Returns:
        Optional[Dict]: Metadata record dictionary if found, else None.
    """
    client = chromadb.PersistentClient(path="chromadb")
    collections = [c.name for c in client.list_collections()]
    
    if collection_name not in collections:
        console.print(f"Collection '{collection_name}' not found", style='bold red')
        return None
        
    coll = client.get_collection(collection_name)
    data = coll.get()
    formatted_id = f"{record_id:03}"
    
    if formatted_id in data["ids"]:
        idx = data["ids"].index(formatted_id)
        return data["metadatas"][idx]
    else:
        console.print(f"Record '{record_id}' ({formatted_id}) does not exist in collection.", style='bold red')
        return None

In [12]:
inspect_game_record('udaplay', 1)

{'Description': "An expansive open-world game set in the fictional state of San Andreas, following the story of Carl 'CJ' Johnson.",
 'Genre': 'Action-adventure',
 'Publisher': 'Rockstar Games',
 'Platform': 'PlayStation 2',
 'YearOfRelease': 2004,
 'Name': 'Grand Theft Auto: San Andreas'}

### 6. Dataset Distribution & Platform Analytics

In [13]:
def retrieve_all_games() -> List[GameDocument]:
    """
    Retrieve all indexed video game records from the vector store.
    
    Returns:
        List[GameDocument]: Full catalog of indexed games as typed models.
    """
    client = chromadb.PersistentClient(path="chromadb")
    coll = client.get_collection("udaplay")
    records = coll.get()
    
    if not records or not records.get("metadatas"):
        return []
        
    return [GameDocument(**meta) for meta in records["metadatas"]]

In [14]:
# Build unified pandas DataFrame from indexed records
catalog_entries = retrieve_all_games()
records_df = pd.DataFrame([entry.model_dump() for entry in catalog_entries])
records_df

,Platform,Name,YearOfRelease,Description
0,PlayStation 1,Gran Turismo,1997,A realistic racing simulator featuring a wide ...
1,PlayStation 2,Grand Theft Auto: San Andreas,2004,An expansive open-world game set in the fictio...
2,PlayStation 3,Gran Turismo 5,2010,A comprehensive racing simulator featuring a v...
3,PlayStation 4,Marvel's Spider-Man,2018,An open-world superhero game that lets players...
4,PlayStation 5,Marvel's Spider-Man 2,2023,"The sequel to the acclaimed Spider-Man game, f..."
5,Game Boy Color,Pokémon Gold and Silver,1999,Second-generation Pokémon games introducing ne...
6,Game Boy Advance,Pokémon Ruby and Sapphire,2002,Third-generation Pokémon games set in the Hoen...
7,Super Nintendo Entertainment System (SNES),Super Mario World,1990,A classic platformer where Mario embarks on a ...
8,Nintendo 64,Super Mario 64,1996,A groundbreaking 3D platformer that set new st...
9,GameCube,Super Smash Bros. Melee,2001,A crossover fighting game featuring characters...


In [15]:
# Cross-tabulate game count by Platform and Year of Release
platform_table = pd.pivot_table(
    records_df,
    index=['Platform'],
    columns=['YearOfRelease'],
    values=['Name'],
    aggfunc='count',
    fill_value=0,
    margins=True,
    margins_name="Total"
)
platform_table.replace(0, '')

Name                                \
YearOfRelease                              1990 1996 1997 1999 2001 2002 2004   
Platform                                                                        
Game Boy Advance                                                       1        
Game Boy Color                                               1                  
GameCube                                                          1             
Nintendo 64                                        1                            
Nintendo Switch                                                                 
PC                                                                              
PlayStation 1                                           1                       
PlayStation 2                                                               1   
PlayStation 3                                                                   
PlayStation 4                                                                   
PlayStation 5                                                                   
Super Nintendo Entertainment System (SNES)    1                                 
Wii                                                                             
Windows                                                                         
Xbox 360                                                                        
Xbox One                                                                        
Xbox Series X/S                                                                 
Xbox Series X|S                                                                 
Total                                         1    1    1    1    1    1    1   

                                                                               \
YearOfRelease                              2006 2010 2014 2017 2018 2021 2022   
Platform                                                                        
Game Boy Advance                                                                
Game Boy Color                                                                  
GameCube                                                                        
Nintendo 64                                                                     
Nintendo Switch                                              1                  
PC                                                                              
PlayStation 1                                                                   
PlayStation 2                                                                   
PlayStation 3                                      1                            
PlayStation 4                                                     1         1   
PlayStation 5                                                               1   
Super Nintendo Entertainment System (SNES)                                      
Wii                                           1                                 
Windows                                                                         
Xbox 360                                           1                            
Xbox One                                                1                       
Xbox Series X/S                                                        1    1   
Xbox Series X|S                                                        1        
Total                                         1    2    1    1    1    2    3   

                                                                 
YearOfRelease                              2023 2024 2025 Total  
Platform                                                         
Game Boy Advance                                              1  
Game Boy Color                                                1  
GameCube                                                      1  
Nintendo 64                                                   1  
Nintendo Switch

### Part 01 Summary
- Initialized persistent **ChromaDB** client at `./chromadb`.
- Implemented vector embedding pipeline with OpenAI embeddings API.
- Successfully indexed 48 video game records across multiple console generations (PS1-PS5, PC, Nintendo, Xbox).
- Defined `GameDocument` Pydantic model for safe data typing.
- The vector store is ready for agent tool integration in Part 02.